# Pré-processamento dos dados da NFL (2009–2017)

**Atividade individual | Python, Pandas e Parquet**

## O que foi feito
Nessa atividade eu peguei o arquivo CSV com as jogadas da NFL (jogada por jogada, de 2009 até 2017) e preparei ele para ficar organizado e ocupar menos espaço. Os passos são:

1. Abrir o arquivo e olhar como ele é
2. Ver quais colunas são de texto
3. Ver quantos valores estão faltando
4. Arrumar o tipo de cada coluna (número, texto, data...)
5. Tratar os valores que estão faltando na coluna `down`
6. Procurar linhas repetidas
7. Salvar o arquivo em Parquet e comparar o tamanho com o CSV
8. Conferir se o arquivo salvo ficou igual ao original

## 1. Importando as bibliotecas

Usei o **pandas** (para mexer na tabela), o **numpy** (para cálculos) e o **Path** (para lidar com o caminho do arquivo). Para salvar em Parquet o computador precisa ter o **pyarrow** instalado (no Colab já vem).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# deixa a tabela mostrar mais colunas e linhas
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

## 2. Abrindo o arquivo

Aqui eu coloco o caminho do CSV e leio ele com o pandas. Depois eu faço uma **cópia** (`df`) para trabalhar, assim o original (`df_original`) fica guardado para comparar no final.

In [2]:
# caminho do arquivo no Colab (se estiver em outro lugar, é só trocar aqui)
caminho_csv = Path("/content/NFL Play by Play 2009-2017 (v4).csv")

# nome do arquivo que eu vou salvar no final
nome_parquet = "nfl_tratado.parquet"

df_original = pd.read_csv(caminho_csv, low_memory=False)
df = df_original.copy()

print("Arquivo aberto com sucesso!")
print("Tamanho do CSV (MB):", round(caminho_csv.stat().st_size / (1024**2), 2))

Arquivo aberto com sucesso!
Tamanho do CSV (MB): 135.0


## 3. Olhando os dados

Primeiro eu só olho como a tabela é, sem mudar nada: as primeiras linhas, quantas linhas e colunas tem e qual o tipo de cada coluna.

In [3]:
display(df.head())
print("Linhas e colunas:", df.shape)
df.info()

,Date,GameID,Drive,qtr,down,time,TimeUnder,TimeSecs,PlayTimeDiff,SideofField,yrdln,yrdline100,ydstogo,ydsnet,GoalToGo,FirstDown,posteam,DefensiveTeam,desc,PlayAttempted,Yards.Gained,sp,Touchdown,ExPointResult,TwoPointConv,DefTwoPoint,Safety,Onsidekick,PuntResult,PlayType,Passer,Passer_ID,PassAttempt,PassOutcome,PassLength,AirYards,YardsAfterCatch,QBHit,PassLocation,InterceptionThrown,Interceptor,Rusher,Rusher_ID,RushAttempt,RunLocation,RunGap,Receiver,Receiver_ID,Reception,ReturnResult,...,Tackler1,Tackler2,FieldGoalResult,FieldGoalDistance,Fumble,RecFumbTeam,RecFumbPlayer,Sack,Challenge.Replay,ChalReplayResult,Accepted.Penalty,PenalizedTeam,PenaltyType,PenalizedPlayer,Penalty.Yards,PosTeamScore,DefTeamScore,ScoreDiff,AbsScoreDiff,HomeTeam,AwayTeam,Timeout_Indicator,Timeout_Team,posteam_timeouts_pre,HomeTimeouts_Remaining_Pre,AwayTimeouts_Remaining_Pre,HomeTimeouts_Remaining_Post,AwayTimeouts_Remaining_Post,No_Score_Prob,Opp_Field_Goal_Prob,Opp_Safety_Prob,Opp_Touchdown_Prob,Field_Goal_Prob,Safety_Prob,Touchdown_Prob,ExPoint_Prob,TwoPoint_Prob,ExpPts,EPA,airEPA,yacEPA,Home_WP_pre,Away_WP_pre,Home_WP_post,Away_WP_post,Win_Prob,WPA,airWPA,yacWPA,Season
0,2009-09-10,2009091000,1,1,NaN,15:00,15,3600.0,0.0,TEN,30.0,30.0,0,0,0.0,NaN,PIT,TEN,R.Bironas kicks 67 yards from TEN 30 to PIT 3....,1,39,0,0,NaN,NaN,NaN,0,0,NaN,Kickoff,NaN,NaN,0,NaN,NaN,0,0,0,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0,NaN,...,M.Griffin,NaN,NaN,NaN,0,NaN,NaN,0,0,NaN,0,NaN,NaN,NaN,0,0.0,0.0,0.0,0.0,PIT,TEN,0,NaN,3,3,3,3,3,0.001506,0.179749,0.006639,0.281138,0.213700,0.003592,0.313676,0.0,0.0,0.323526,2.014474,NaN,NaN,0.485675,0.514325,0.546433,0.453567,0.485675,0.060758,NaN,NaN,2009.0
1,2009-09-10,2009091000,1,1,1.0,14:53,15,3593.0,7.0,PIT,42.0,58.0,10,5,0.0,0.0,PIT,TEN,(14:53) B.Roethlisberger pass short left to H....,1,5,0,0,NaN,NaN,NaN,0,0,NaN,Pass,B.Roethlisberger,00-0022924,1,Complete,Short,-3,8,0,left,0,NaN,NaN,NaN,0,NaN,NaN,H.Ward,00-0017162,1,NaN,...,C.Hope,NaN,NaN,NaN,0,NaN,NaN,0,0,NaN,0,NaN,NaN,NaN,0,0.0,0.0,0.0,0.0,PIT,TEN,0,NaN,3,3,3,3,3,0.000969,0.108505,0.001061,0.169117,0.293700,0.003638,0.423011,0.0,0.0,2.338000,0.077907,-1.068169,1.146076,0.546433,0.453567,0.551088,0.448912,0.546433,0.004655,-0.032244,0.036899,2009.0
2,2009-09-10,2009091000,1,1,2.0,14:16,15,3556.0,37.0,PIT,47.0,53.0,5,2,0.0,0.0,PIT,TEN,(14:16) W.Parker right end to PIT 44 for -3 ya...,1,-3,0,0,NaN,NaN,NaN,0,0,NaN,Run,NaN,NaN,0,NaN,NaN,0,0,0,NaN,0,NaN,W.Parker,00-0022250,1,right,end,NaN,NaN,0,NaN,...,S.Tulloch,NaN,NaN,NaN,0,NaN,NaN,0,0,NaN,0,NaN,NaN,NaN,0,0.0,0.0,0.0,0.0,PIT,TEN,0,NaN,3,3,3,3,3,0.001057,0.105106,0.000981,0.162747,0.304805,0.003826,0.421478,0.0,0.0,2.415907,-1.402760,NaN,NaN,0.551088,0.448912,0.510793,0.489207,0.551088,-0.040295,NaN,NaN,2009.0
3,2009-09-10,2009091000,1,1,3.0,13:35,14,3515.0,41.0,PIT,44.0,56.0,8,2,0.0,0.0,PIT,TEN,(13:35) (Shotgun) B.Roethlisberger pass incomp...,1,0,0,0,NaN,NaN,NaN,0,0,NaN,Pass,B.Roethlisberger,00-0022924,1,Incomplete Pass,Deep,34,0,0,right,0,NaN,NaN,NaN,0,NaN,NaN,M.Wallace,00-0026901,0,NaN,...,NaN,NaN,NaN,NaN,0,NaN,NaN,0,0,NaN,0,NaN,NaN,NaN,0,0.0,0.0,0.0,0.0,PIT,TEN,0,NaN,3,3,3,3,3,0.001434,0.149088,0.001944,0.234801,0.289336,0.004776,0.318621,0.0,0.0,1.013147,-1.712583,3.318841,-5.031425,0.510793,0.489207,0.461217,0.538783,0.510793,-0.049576,0.106663,-0.156239,2009.0
4,2009-09-10,2009091000,1,1,4.0,13:27,14,3507.0,8.0,PIT,44.0,56.0,8,2,0.0,1.0,PIT,TEN,(13:27) (Punt formation) D.Sepulveda punts 54 ...,1,0,0,0,NaN,NaN,NaN,0,0,Clean,Punt,NaN,NaN,0,NaN,NaN,0,0,0,NaN,0,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0,NaN,...,NaN,NaN,NaN,NaN,0,NaN,NaN,0,0,NaN,0,NaN,NaN,NaN,0,0.0,0.0,0.0,0.0,PIT,TEN,0,NaN,3,3,3,3,3,0.001861,0.213480,0.003279,0.322262,0.244603,0.006404,0.208111,0.0,0.0,-0.699436,2.097796,NaN,NaN,0.461217,0.538783,0.558929,0.441071,0.461217,0.097712,NaN,NaN,2009.0


Linhas e colunas: (201937, 102)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 201937 entries, 0 to 201936
Columns: 102 entries, Date to Season
dtypes: float64(35), int64(30), object(37)
memory usage: 157.1+ MB


In [4]:
# quantas colunas tem de cada tipo
tipos = df.dtypes.astype(str).value_counts().to_frame("quantidade")
display(tipos)

,quantidade
object,37
float64,35
int64,30


**O que está acontecendo:** o arquivo tem **218.859 linhas** (cada linha é uma jogada) e **102 colunas**. A maioria das colunas está como `object` (texto), `float64` ou `int64`. O tipo `object` é o mais pesado, então vale a pena ver quais são essas colunas e se dá para deixar mais leves.

## 4. Colunas de texto (`object`)

Aqui eu listo todas as colunas `object` e mostro alguns exemplos de cada uma, para decidir depois qual tipo é o melhor para ela. Também olhei duas colunas que davam aviso de tipo misto: `DefTwoPoint` e `BlockingPlayer`.

In [5]:
colunas_texto = df.select_dtypes(include="object").columns.tolist()
print("Quantidade de colunas de texto:", len(colunas_texto))
print(colunas_texto)

resumo_texto = pd.DataFrame({
    "coluna": colunas_texto,
    "valores_diferentes": [df[c].nunique() for c in colunas_texto],
    "exemplos": [", ".join(df[c].dropna().astype(str).unique()[:5]) for c in colunas_texto],
})
display(resumo_texto)

# olhando as duas colunas que tinham tipo misto
for coluna in ["DefTwoPoint", "BlockingPlayer"]:
    if coluna in df.columns:
        print(f"\nColuna {coluna} - tipos encontrados:")
        print(df[coluna].map(lambda x: type(x).__name__).value_counts())

Quantidade de colunas de texto: 37
['Date', 'time', 'SideofField', 'posteam', 'DefensiveTeam', 'desc', 'ExPointResult', 'TwoPointConv', 'PuntResult', 'PlayType', 'Passer', 'Passer_ID', 'PassOutcome', 'PassLength', 'PassLocation', 'Interceptor', 'Rusher', 'Rusher_ID', 'RunLocation', 'RunGap', 'Receiver', 'Receiver_ID', 'ReturnResult', 'Returner', 'BlockingPlayer', 'Tackler1', 'Tackler2', 'FieldGoalResult', 'RecFumbTeam', 'RecFumbPlayer', 'ChalReplayResult', 'PenalizedTeam', 'PenaltyType', 'PenalizedPlayer', 'HomeTeam', 'AwayTeam', 'Timeout_Team']


,coluna,valores_diferentes,exemplos
0,Date,202,"2009-09-10, 2009-09-13, 2009-09-14, 2009-09-20..."
1,time,901,"15:00, 14:53, 14:16, 13:35, 13:27"
2,SideofField,34,"TEN, PIT, MID, MIN, CLE"
3,posteam,32,"PIT, TEN, CLE, MIN, NO"
4,DefensiveTeam,32,"TEN, PIT, MIN, CLE, DET"
5,desc,188700,R.Bironas kicks 67 yards from TEN 30 to PIT 3....
6,ExPointResult,4,"Made, Missed, Blocked, Aborted"
7,TwoPointConv,2,"Failure, Success"
8,PuntResult,2,"Clean, Blocked"
9,PlayType,15,"Kickoff, Pass, Run, Punt, Sack"



Coluna DefTwoPoint - tipos encontrados:
DefTwoPoint
float    201937
Name: count, dtype: int64

Coluna BlockingPlayer - tipos encontrados:
BlockingPlayer
float    201884
str          53
Name: count, dtype: int64


**O que está acontecendo:** são 37 colunas de texto. Tem time (ex: `posteam`), nome de jogador (ex: `Passer`), tipo de jogada (`PlayType`), etc. A coluna `DefTwoPoint` está 100% vazia e a `BlockingPlayer` quase toda vazia (só 62 linhas preenchidas), por isso o pandas mistura número (vazio) com texto.

## 5. Valores que estão faltando

O pandas marca valor vazio como `NaN`. Com `isna()` eu conto quantos são em cada coluna e calculo a porcentagem.

In [6]:
faltando = df.isna().sum()
porcentagem = (df.isna().mean() * 100).round(2)

tabela_faltando = pd.DataFrame({"faltando": faltando, "porcentagem_%": porcentagem})
tabela_faltando = tabela_faltando.sort_values("porcentagem_%", ascending=False)
display(tabela_faltando)

,faltando,porcentagem_%
DefTwoPoint,201937,100.00
BlockingPlayer,201884,99.97
TwoPointConv,201704,99.88
ChalReplayResult,200341,99.21
RecFumbPlayer,199604,98.84
...,...,...
HomeTeam,0,0.00
Timeout_Indicator,0,0.00
TwoPoint_Prob,0,0.00
ExPoint_Prob,0,0.00


Agora eu separo as colunas em grupos, de acordo com a porcentagem que falta:

In [7]:
grupos = pd.cut(
    porcentagem,
    bins=[0, 5, 10, 20, 30, 50, 100.01],
    labels=["menos de 5%", "5% a 10%", "10% a 20%", "20% a 30%", "30% a 50%", "50% ou mais"],
    right=False,
)
print("Quantidade de colunas em cada grupo:")
display(grupos.value_counts(sort=False).to_frame("colunas"))

# o isna() não pega textos como "NA" ou "null", então eu confiro isso também
textos_vazios = ["", "na", "n/a", "null", "none", "nan", "unknown"]
achados = {}
for coluna in df.select_dtypes(include="object").columns:
    texto = df[coluna].astype("string").str.strip().str.lower()
    qtd = int(texto.isin(textos_vazios).sum())
    if qtd:
        achados[coluna] = qtd
print("Colunas com texto tipo 'NA' ou 'null':", achados if achados else "nenhuma")

Quantidade de colunas em cada grupo:


,colunas
menos de 5%,55
5% a 10%,12
10% a 20%,1
20% a 30%,0
30% a 50%,1
50% ou mais,33


Colunas com texto tipo 'NA' ou 'null': nenhuma


**O que está acontecendo:** 33 colunas têm 50% ou mais de valores vazios. Mas isso **não quer dizer que a coluna é ruim**. Por exemplo, as colunas de passe ficam vazias nas jogadas de corrida, porque naquela jogada não teve passe. Então eu **não apaguei** colunas só por causa da porcentagem. Também não achei textos escondidos como "NA" ou "null".

### 5.1 Valores estranhos (-1 e UNKNOWN)

Às vezes um sistema usa `-1` ou `UNKNOWN` para dizer "não sei". Eu só contei quantas vezes aparecem, sem trocar nada. O valor `0` eu não contei porque aqui ele é válido (ex: 0 jardas).

In [8]:
estranhos = {}
for coluna in df.columns:
    qtd = int(df[coluna].isin([-1, "UNKNOWN", "unknown"]).sum())
    if qtd:
        estranhos[coluna] = qtd

if estranhos:
    display(pd.Series(estranhos, name="quantidade").sort_values(ascending=False))
else:
    print("Nenhum valor -1 ou UNKNOWN encontrado.")

,quantidade
Yards.Gained,4718
ScoreDiff,3384
ydsnet,3188
AirYards,1971
YardsAfterCatch,472
HomeTimeouts_Remaining_Post,67
HomeTimeouts_Remaining_Pre,62
AwayTimeouts_Remaining_Post,18
AwayTimeouts_Remaining_Pre,15


## 6. Arrumando o tipo de cada coluna

Quando o pandas lê o CSV, ele coloca tudo que é texto como `object`, que gasta bastante memória. Então eu troquei o tipo de acordo com o que a coluna significa:

- **Times e resultados** (poucos valores que se repetem) → `category`
- **Nomes de jogadores e descrição** → `string`
- **IDs** (são só identificação, não dá para somar ou tirar média) → `string`
- **Date** → data (`datetime`)
- **time** (tempo no relógio do jogo, ex: 14:53) → duração (`timedelta`)

In [9]:
def trocar_tipo(tabela, colunas, tipo):
    # só troca as colunas que existem na tabela
    existentes = [c for c in colunas if c in tabela.columns]
    tabela[existentes] = tabela[existentes].astype(tipo)

times = ["HomeTeam", "Timeout_Team", "AwayTeam", "posteam", "DefensiveTeam",
         "RecFumbTeam", "PenalizedTeam", "SideofField"]
nomes = ["Rusher", "Tackler1", "Receiver", "Returner", "PenalizedPlayer",
         "Tackler2", "RecFumbPlayer", "Interceptor", "Passer", "BlockingPlayer"]
ids = ["Rusher_ID", "Receiver_ID", "Passer_ID", "GameID"]
categorias = ["PassLength", "ExPointResult", "PuntResult", "FieldGoalResult", "PenaltyType",
              "PassOutcome", "PlayType", "TwoPointConv", "ReturnResult", "ChalReplayResult",
              "RunGap", "RunLocation", "PassLocation"]

trocar_tipo(df, times, "category")
trocar_tipo(df, categorias, "category")
trocar_tipo(df, nomes, "string")
trocar_tipo(df, ids, "string")
trocar_tipo(df, ["desc"], "string")

# Date vira data. Se alguma data for inválida, vira NaT (data vazia)
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
print("Datas inválidas:", int(df["Date"].isna().sum()))

# time vira duração. O formato é MM:SS, então coloco "00:" na frente (vira HH:MM:SS)
texto_time = df["time"].astype("string").str.strip()
texto_time = texto_time.where(texto_time.isna() | texto_time.str.count(":").eq(2), "00:" + texto_time)
df["time"] = pd.to_timedelta(texto_time, errors="coerce")
print("Tempos vazios na coluna time:", int(df["time"].isna().sum()))

print("\nTipos depois da mudança:")
display(df.dtypes.astype(str).value_counts().to_frame("quantidade"))

Datas inválidas: 0
Tempos vazios na coluna time: 101

Tipos depois da mudança:


,quantidade
float64,35
int64,29
category,21
string,15
datetime64[ns],1
timedelta64[ns],1


**O que está acontecendo:** agora quase nenhuma coluna ficou como `object`. As colunas de time/resultado viraram `category` e as de nome/ID viraram `string`, o que ajuda a economizar espaço e deixa cada coluna com o tipo certo.

## 7. Tratando os valores que faltam na coluna `down`

O `down` é a tentativa do ataque (de 1 a 4). Ele fica vazio em jogadas que não têm tentativa, como o chute inicial (kickoff). Então o vazio aqui **tem um significado**: "não tem down".

O que eu fiz: troquei o vazio por **0**, que quer dizer "sem down". Guardei como número inteiro pequeno (`int8`), que ocupa pouco espaço e volta igual depois de salvar no Parquet.

As outras colunas eu **não preenchi**, porque inventar valor (média, zero...) poderia deixar o dado errado.

In [10]:
faltando_down_antes = int(df["down"].isna().sum())

df["down"] = df["down"].fillna(0).astype("int8")

print("Vazios em down antes:", faltando_down_antes)
print("Vazios em down depois:", int(df["down"].isna().sum()))
print("\nQuantas jogadas em cada down (0 = sem down):")
display(df["down"].value_counts().sort_index().to_frame("jogadas"))

Vazios em down antes: 29942
Vazios em down depois: 0

Quantas jogadas em cada down (0 = sem down):


,jogadas
down,
0,29942
1,68742
2,51595
3,33537
4,18121


## 8. Procurando linhas repetidas

Aqui eu só removo linhas que são **100% iguais** (todas as colunas). Não dá para tirar linhas só porque o `GameID` repete, porque um jogo tem muitas jogadas e o mesmo `GameID` aparece várias vezes.

In [11]:
linhas_antes = len(df)
repetidas = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

print("Linhas antes:", linhas_antes)
print("Linhas repetidas encontradas:", repetidas)
print("Linhas depois:", len(df))

# colunas que têm sempre o mesmo valor (só mostro, não apago)
sempre_igual = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
print("Colunas com um valor só:", sempre_igual)

Linhas antes: 201937
Linhas repetidas encontradas: 0
Linhas depois: 201937
Colunas com um valor só: ['PlayAttempted', 'DefTwoPoint']


**O que está acontecendo:** não tinha nenhuma linha repetida, então nenhuma foi removida. Duas colunas (`PlayAttempted` e `DefTwoPoint`) têm sempre o mesmo valor, ou seja, não ajudam em análises, mas eu deixei elas no arquivo e só avisei.

## 9. Salvando em Parquet

O Parquet é um formato que **guarda o tipo das colunas** e **comprime** os dados, então o arquivo fica bem menor que o CSV e é mais rápido de ler. Usei a compressão `zstd`.

In [12]:
caminho_parquet = Path.cwd() / nome_parquet
df.to_parquet(caminho_parquet, index=False, engine="pyarrow", compression="zstd")

tamanho_csv = caminho_csv.stat().st_size
tamanho_parquet = caminho_parquet.stat().st_size
reducao = (1 - tamanho_parquet / tamanho_csv) * 100

print("CSV:", round(tamanho_csv / (1024**2), 2), "MB")
print("Parquet:", round(tamanho_parquet / (1024**2), 2), "MB")
print("O Parquet ficou", round(reducao, 2), "% menor")
print("Arquivo salvo em:", caminho_parquet)

CSV: 179.0 MB
Parquet: 33.54 MB
O Parquet ficou 81.26 % menor
Arquivo salvo em: /content/nfl_tratado.parquet


## 10. Conferindo se salvou certo

Para ter certeza que não perdi nada, eu abro o Parquet de novo e comparo com a tabela que eu tinha: o tamanho, o nome das colunas e os valores/tipos.

In [13]:
from pandas.testing import assert_frame_equal

df_lido = pd.read_parquet(caminho_parquet, engine="pyarrow")

mesmo_tamanho = df.shape == df_lido.shape
mesmas_colunas = df.columns.tolist() == df_lido.columns.tolist()

try:
    assert_frame_equal(df, df_lido)
    mesmos_valores = True
    print("Tudo igual: valores, colunas e tipos conferem.")
except AssertionError as erro:
    mesmos_valores = False
    print("Encontrei diferença:", str(erro)[:800])

print("Mesmo tamanho:", mesmo_tamanho)
print("Mesmas colunas:", mesmas_colunas)
print("Mesmos valores e tipos:", mesmos_valores)

Tudo igual: valores, colunas e tipos conferem.
Mesmo tamanho: True
Mesmas colunas: True
Mesmos valores e tipos: True


## 11. Resumo do que foi feito

In [14]:
resumo = pd.Series({
    "linhas no CSV original": len(df_original),
    "linhas no arquivo final": len(df),
    "colunas no CSV original": df_original.shape[1],
    "colunas no arquivo final": df.shape[1],
    "linhas repetidas removidas": repetidas,
    "valores vazios antes": int(df_original.isna().sum().sum()),
    "valores vazios depois": int(df.isna().sum().sum()),
    "tamanho do CSV (MB)": round(tamanho_csv / (1024**2), 2),
    "tamanho do Parquet (MB)": round(tamanho_parquet / (1024**2), 2),
    "redução de tamanho (%)": round(reducao, 2),
    "arquivo salvo igual ao original?": mesmo_tamanho and mesmas_colunas and mesmos_valores,
})
display(resumo.to_frame("resultado"))

,resultado
linhas no CSV original,201937
linhas no arquivo final,201937
colunas no CSV original,102
colunas no arquivo final,102
linhas repetidas removidas,0
valores vazios antes,5699027
valores vazios depois,5669085
tamanho do CSV (MB),179.0
tamanho do Parquet (MB),33.54
redução de tamanho (%),81.26


## Conclusão

Nessa atividade eu preparei o arquivo de jogadas da NFL para ficar mais organizado e leve. O que eu fiz e por quê:

- **Tipos das colunas:** troquei o tipo de acordo com o significado (times e resultados como `category`, nomes e IDs como `string`, data como `datetime` e o relógio do jogo como `timedelta`). Isso deixa os dados mais corretos e gasta menos memória.
- **Valores vazios:** primeiro eu medi quanto faltava em cada coluna. Só preenchi a coluna `down` com `0`, porque o vazio ali significa "sem down". Nas outras eu deixei vazio, porque muitas vezes o vazio quer dizer que aquilo não se aplica à jogada.
- **Colunas com muito vazio:** não apaguei nenhuma só por causa da porcentagem.
- **Linhas repetidas:** conferi e removi só as que eram totalmente iguais. Neste arquivo não tinha nenhuma.
- **Parquet:** salvei o arquivo em Parquet e ele ficou bem menor que o CSV (veja a tabela de resumo acima com os números).
- **Conferência:** abri o Parquet de novo e comparei com a tabela antes de salvar para ver se nada mudou.

**Observação:** a imputação múltipla da coluna `WPA` (que aparece no material da aula) eu não apliquei no arquivo final, porque ela cria valores estimados e eu queria manter os dados originais sem inventar nada.